Prompt --> LLM -------(Generates Query) ----> Hit Sql Database ---> context -----> LLM ---> response

In [4]:
import sqlite3
from pathlib import Path
import os

In [2]:
# connect to sqlite database

connection = sqlite3.connect('my_database.db')

In [7]:
# create a cursor object (responsible to inserting, retreiving, updating, create tables etc)

cursor = connection.cursor()

In [5]:
# create the table if it doesn't exists
table_info = """
CREATE TABLE IF NOT EXISTS student_info (
Name VARCHAR,
Class VARCHAR,
Section VARCHAR,
Roll_No INT,
Marks INT
);
"""

In [8]:
cursor.execute(table_info)

In [10]:
# Insert some more rescords

cursor.execute("INSERT INTO student_info (Name, Class, Section, Roll_No, Marks) VALUES ('John Doe', '10th', 'A', 1, 85)," \
"('Jane Smith', '10th', 'B', 2, 90)," \
"('Michael Johnson', '10th', 'A', 3, 78)," \
"('Emily Davis', '10th', 'B', 4, 92)" )

In [11]:
# Display the records in the table
print("Records in the table:")
Data = cursor.execute("SELECT * FROM student_info")
for row in Data:
    print(row)

Records in the table:
('John Doe', '10th', 'A', 1, 85)
('Jane Smith', '10th', 'B', 2, 90)
('Michael Johnson', '10th', 'A', 3, 78)
('Emily Davis', '10th', 'B', 4, 92)


In [12]:
# close the connection to the database
connection.commit()
connection.close()

In [13]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
 )

`sqlite_master` is SQLite's internal catalog table — every database has one automatically. It stores a row describing each object in the database. It's SQLite's equivalent of `INFORMATION_SCHEMA` in SQL Server.

Its columns:

| column | holds |
|---|---|
| `type` | `table`, `index`, `view`, or `trigger` |
| `name` | the object's name |
| `tbl_name` | the table it belongs to |
| `rootpage` | internal storage detail |
| `sql` | **the original CREATE statement, as text** |

So the query reads: *give me the `sql` column for every row where the object is a table and its name doesn't start with `sqlite_`*.

- `type='table'` — skip indexes, views, triggers
- `name NOT LIKE 'sqlite_%'` — skip SQLite's own internal bookkeeping tables, which you don't want in the prompt
- selecting `sql` — returns the literal `CREATE TABLE claims (...)` text that was used to create it

Run it yourself to see:

```python
conn = sqlite3.connect(DB_PATH)
for row in conn.execute("SELECT type, name, sql FROM sqlite_master"):
    print(row)
```

The `for (sql,) in cur.execute(...)` syntax is unpacking — each row comes back as a one-element tuple like `('CREATE TABLE claims (...)',)`, and the trailing comma pulls the string out.

The reason this approach beats typing the schema by hand: it's always accurate. If the database you were given differs from what you assumed, or you look at it again in two days, the prompt reflects what's actually there rather than what you remembered.

In [30]:
def get_schema(db_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    parts = []
    for (sql,) in cur.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    ):
        parts.append(sql)
    conn.close()
    return "\n\n".join(parts)

SCHEMA = get_schema("my_database.db")
print(SCHEMA)

CREATE TABLE student_info (
Name VARCHAR,
Class VARCHAR,
Section VARCHAR,
Roll_No INT,
Marks INT
)


In [37]:
def generate_sql_query(question: str, schema: str) -> str:
    """
    Generate SQL query from a natural language question using Groq API.

    Args:
        question (str): The natural language question to convert into SQL.
        schema (str): The database schema.

    """

    gen_prompt = f"""
you are an expert SQL developer. You are given a natural language question and a database schema. Your task is to write a single SQL query that answers the question based on the provided schema.
Write a single SQL query answering: {question}
My database schema is :
{schema}
Return only the SQL, no explanation. SQL code should not have ``` in the begining or end of the code block. Return only the SQL query, no other text or explanation."""

    response = llm.invoke(gen_prompt)
    return response.content

In [32]:
response = generate_sql_query("What is the average marks of students in class 10th?", SCHEMA)

In [33]:
print(response)

SELECT AVG(Marks) AS avg_marks
FROM student_info
WHERE Class = '10th';


In [34]:
def read_sql_query(sql,db):
    """
    Execute a SQL query on a SQLite database and return the results.

    Args:
        sql (str): The SQL query to execute.
        db (str): The path to the SQLite database file.
    """

    connection = sqlite3.connect(db)
    cursor = connection.cursor()
    cursor.execute(sql)
    rows = cursor.fetchall()
    connection.commit()
    connection.close()
    return rows


In [35]:
rows = read_sql_query(response, 'my_database.db')

In [36]:
for row in rows:
    print(row)

(86.25,)


In [38]:
def final_llm_response(question: str, db_path: str) -> str:
    """
    Generate SQL query from a natural language question using Groq API and execute it on the database.

    Args:
        question (str): The natural language question to convert into SQL.
        db_path (str): The path to the SQLite database file.
    """

    schema = get_schema(db_path)
    sql_query = generate_sql_query(question, schema)
    rows = read_sql_query(sql_query, db_path)
    prompt = f"""
You are an expert SQL developer. You are given a natural language question, a database schema, and the results of a SQL query executed on the database.
Your task is to provide a concise and accurate answer to the question based on the provided schema and query results.

User question : {question}
Database schema : {schema}
SQL query results : {rows}
Provide a concise and accurate answer to the user's question based on the provided schema and query results.
""" 

    response = llm.invoke(prompt)
    return response.content


In [39]:
answer = final_llm_response("What is the average marks of students in class 10th?", 'my_database.db')
print(answer)

The average marks of students in class 10th is **86.25**.


**MindSQL Architecture For mediassist.db**


In [40]:
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

client = QdrantClient(":memory:")
embedder = TextEmbedding("BAAI/bge-small-en-v1.5")

COLLECTION = "mindsql_demo"

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)

client.create_payload_index(
    collection_name=COLLECTION,
    field_name="type",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

c:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 2\Coding\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 2\Coding\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kilar\AppData\Local\Temp\fastembed_cache. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support s

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [42]:
from pathlib import Path

DB_PATH = Path.cwd().parent / "Data" / "db" / "mediassist.db"
print(DB_PATH, DB_PATH.exists())

c:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 2\Coding\Data\db\mediassist.db True


In [41]:
import uuid

def _add(text: str, payload: dict):
    vec = list(embedder.embed([text]))[0].tolist()
    client.upsert(
        collection_name=COLLECTION,
        points=[models.PointStruct(
            id=str(uuid.uuid4()),
            vector=vec,
            payload=payload,
        )],
    )

def train_ddl(ddl: str):
    _add(ddl, {"type": "ddl", "content": ddl})

def train_documentation(doc: str):
    _add(doc, {"type": "doc", "content": doc})

def train_sql(question: str, sql: str):
    # embed the QUESTION, store the SQL alongside
    _add(question, {"type": "sql", "question": question, "sql": sql})

In [43]:
import sqlite3

conn = sqlite3.connect(DB_PATH)
for (sql,) in conn.execute(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
):
    train_ddl(sql)
conn.close()

print(client.count(COLLECTION))

count=2
